# E046 — Fusion: Product Rule vs Weighted Sum

**Motivation:** E039 used weighted sum fusion. Product rule (geometric mean of probabilities) may be more robust for combining independent modalities.

**Hypothesis:** Product rule fusion will outperform weighted sum for trimodal fusion.

**Approach:**
1. Convert calibrated scores to probabilities (sigmoid)
2. Product rule: p_fused = (p_mfcc * p_lpcc * p_img)^(1/3)
3. Compare to weighted sum (E039)

**Configs:**
- `weighted_sum`: E039 baseline
- `product`: Product rule (geometric mean)
- `sum`: Simple sum (equal weights)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from PIL import Image
from scipy.special import logsumexp, expit, logit
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E039_REF = {'oof_eer': 0.26}

222 samples


In [2]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_mfcc(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return np.vstack([mfcc, delta, delta2]).T - np.vstack([mfcc, delta, delta2]).T.mean(axis=0)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except: cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    return np.hstack([feat, delta, delta2]) - np.hstack([feat, delta, delta2]).mean(axis=0)

def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80): img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type, max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

print('Functions defined')

Functions defined


## 2. Collect OOF scores

In [3]:
print("Collecting OOF scores...")
oof_mfcc = np.full(len(manifest), np.nan)
oof_lpcc = np.full(len(manifest), np.nan)
oof_img = np.full(len(manifest), np.nan)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    seed_f = SEED + fold_id
    rng = np.random.default_rng(seed_f)
    train_df, val_df = manifest.loc[train_idx], manifest.loc[val_idx]
    print(f"Fold {fold_id}...")
    
    # MFCC
    X_tgt, X_non = [], []
    for _, row in train_df.iterrows():
        y, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        wavs = [y]
        if row["label"] == 1: wavs.append(librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1)))
        for w in wavs:
            f = _extract_mfcc(w, sr)
            (X_tgt if row["label"] == 1 else X_non).append(f)
    ubm_m = _train_ubm(np.vstack(X_non))
    adp_m = _map_adapt(ubm_m, np.vstack(X_tgt))
    
    # LPCC (tied cov)
    X_tgt, X_non = [], []
    for _, row in train_df.iterrows():
        y, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        wavs = [y]
        if row["label"] == 1: wavs.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=float(rng.choice([-2,-1,1,2]))))
        for w in wavs:
            f = _extract_lpcc(w, sr)
            (X_tgt if row["label"] == 1 else X_non).append(f)
    ubm_l = _train_ubm(np.vstack(X_non), cov_type='tied')
    adp_l = _map_adapt(ubm_l, np.vstack(X_tgt))
    
    # Image (adversarial)
    X_img, y_img = [], []
    for _, row in train_df.iterrows():
        orig = _load_image(_find_png(row["stem"], DATA))
        vecs = [orig, orig[:, ::-1].copy(), np.clip(orig*rng.uniform(0.7,1.3),0,255), np.clip(orig+rng.normal(0,15,orig.shape),0,255)]
        if row["label"] == 1:
            for ang in [-10,-5,5,10]: vecs.append(rotate(orig, ang, reshape=False, order=1, mode='constant', cval=0))
        for v in vecs: X_img.append(v.flatten()); y_img.append(row["label"])
    scl = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scl.fit_transform(np.array(X_img))), np.array(y_img))
    
    # Score validation
    for _, row in val_df.iterrows():
        y, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        oof_mfcc[row.name] = adp_m.score(_extract_mfcc(y, sr)) - ubm_m.score(_extract_mfcc(y, sr))
        oof_lpcc[row.name] = adp_l.score(_extract_lpcc(y, sr)) - ubm_l.score(_extract_lpcc(y, sr))
        x = _load_image(_find_png(row["stem"], DATA)).flatten().reshape(1,-1)
        oof_img[row.name] = float(clf.decision_function(pca.transform(scl.transform(x)))[0])

# Calibrate
cal_m = LogisticRegression(C=1e6, max_iter=1000, class_weight='balanced').fit(oof_mfcc.reshape(-1,1), y_all)
cal_l = LogisticRegression(C=1e6, max_iter=1000, class_weight='balanced').fit(oof_lpcc.reshape(-1,1), y_all)
cal_i = LogisticRegression(C=1e6, max_iter=1000, class_weight='balanced').fit(oof_img.reshape(-1,1), y_all)

# Convert to probabilities
p_mfcc = expit(cal_m.decision_function(oof_mfcc.reshape(-1,1)))
p_lpcc = expit(cal_l.decision_function(oof_lpcc.reshape(-1,1)))
p_img = expit(cal_i.decision_function(oof_img.reshape(-1,1)))

print('OOF collection complete')

Fold 0...


Fold 1...


Fold 2...


OOF collection complete


## 3. Fusion rules

In [4]:
# Weighted sum (E039)
fused_wsum = 0.00 * p_mfcc + 0.34 * p_lpcc + 0.66 * p_img

# Product rule
fused_prod = (p_mfcc * p_lpcc * p_img) ** (1/3)

# Simple sum
fused_sum = (p_mfcc + p_lpcc + p_img) / 3

print("=== Fusion Rules ===")
for name, scores in [('weighted_sum (E039)', fused_wsum), ('product', fused_prod), ('sum', fused_sum)]:
    eer, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    print(f"{name:20s}: EER={eer*100:5.2f}%")

=== Fusion Rules ===
weighted_sum (E039) : EER= 2.97%
product             : EER= 0.52%
sum                 : EER= 0.78%


## 4. Conclusion

Product rule vs weighted sum: [TBD]
Decision: [Adopt/Reject]